# 🍏 Basic Retrieval-Augmented Generation (RAG) with AIProjectClient 🍎

In this notebook, we'll demonstrate a **basic RAG** flow using:
- **`azure-ai-projects`** (AIProjectClient)
- **`azure-ai-inference`** (Embeddings, ChatCompletions)
- **`azure-ai-search`** (for vector or hybrid search)

Our theme is **Health & Fitness** 🍏 so we’ll create a simple set of health tips, embed them, store them in a search index, then do a query that retrieves relevant tips, and pass them to an LLM to produce a final answer.

> **Disclaimer**: This is not medical advice. For real health questions, consult a professional.

## What is RAG?
Retrieval-Augmented Generation (RAG) is a technique where the LLM (Large Language Model) uses relevant retrieved text chunks from your data to craft a final answer. This helps ground the model's response in real data, reducing hallucinations.


<img src="./seq-diagrams/3-basic-rag.png" width="30%"/>

## 1. Setup
We'll import libraries, load environment variables, and create an `AIProjectClient`.

> #### Complete [2-embeddings.ipynb](2-embeddings.ipynb) notebook before starting this one


In [ ]:
import os
import time
import json
import requests
from dotenv import load_dotenv
from pathlib import Path
from urllib.parse import urlparse

from azure.ai.projects import AIProjectClient
from azure.identity import AzureCliCredential
from azure.ai.inference import EmbeddingsClient, ChatCompletionsClient
from azure.ai.inference.models import UserMessage, SystemMessage
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.core.credentials import AzureKeyCredential
from openai import OpenAI

# Load environment variables from workspace root .env
notebook_path = Path().absolute()
load_dotenv(notebook_path.parent.parent / '.env')

# Initialize credentials using AzureCliCredential
credential = AzureCliCredential()

# Parse PROJECT_ENDPOINT into required AIProjectClient constructor components
_url            = os.getenv("PROJECT_ENDPOINT")
_parsed         = urlparse(_url)
base_endpoint   = f"{_parsed.scheme}://{_parsed.netloc}"
path_parts      = [p for p in _parsed.path.split("/") if p]
project_name    = path_parts[-1] if path_parts else ""
hub_name        = _parsed.netloc.split(".")[0]
chat_model      = os.getenv("MODEL_DEPLOYMENT_NAME", "gpt-5.4")
embedding_model = os.getenv("EMBEDDING_MODEL_DEPLOYMENT_NAME")
search_index_name = os.getenv("SEARCH_INDEX_NAME", "healthtips-index")

# Auto-detect subscription_id & resource_group from Foundry hub
print("Auto-detecting subscription ID and resource group...")
try:
    mgmt_token = credential.get_token("https://management.azure.com/.default").token
    headers = {"Authorization": f"Bearer {mgmt_token}"}

    # List all accessible subscriptions
    subs = requests.get(
        "https://management.azure.com/subscriptions?api-version=2020-01-01",
        headers=headers, timeout=15
    ).json().get("value", [])

    subscription_id = None
    resource_group = None
    
    for sub in subs:
        sub_id = sub["subscriptionId"]
        # Search for the Foundry hub: type=Microsoft.CognitiveServices/accounts, name=hub_name
        resources = requests.get(
            f"https://management.azure.com/subscriptions/{sub_id}/resources"
            f"?$filter=name eq '{hub_name}' and "
            f"resourceType eq 'Microsoft.CognitiveServices/accounts'"
            f"&api-version=2021-04-01",
            headers=headers, timeout=15
        ).json().get("value", [])

        if resources:
            # ARM resource ID: /subscriptions/<sub>/resourceGroups/<rg>/providers/...
            rg_from_id = resources[0]["id"].split("/")[4]
            subscription_id = sub_id
            resource_group = rg_from_id
            break

    if not (subscription_id and resource_group):
        raise RuntimeError(f"Hub '{hub_name}' not found in any accessible subscription.")

    print(f"✓ Subscription ID:  {subscription_id[:8]}...")
    print(f"✓ Resource group:   {resource_group}")

except Exception as e:
    raise EnvironmentError(
        f"Failed to auto-detect subscription and resource group: {e}"
    ) from e

try:
    project_client = AIProjectClient(
        endpoint=base_endpoint,
        subscription_id=subscription_id,
        resource_group_name=resource_group,
        project_name=project_name,
        credential=credential,
    )
    print("✅ AIProjectClient created successfully!")
except Exception as e:
    print("❌ Error creating AIProjectClient:", e)

## 2. Create Sample Health Data
We'll create a few short doc chunks. In a real scenario, you might read from CSV or PDFs, chunk them up, embed them, and store them in your search index.


In [ ]:
health_tips = [
    {
        "id": "doc1",
        "content": "Daily 30-minute walks help maintain a healthy weight and reduce stress.",
        "source": "General Fitness"
    },
    {
        "id": "doc2",
        "content": "Stay hydrated by drinking 8-10 cups of water per day.",
        "source": "General Fitness"
    },
    {
        "id": "doc3",
        "content": "Consistent sleep patterns (7-9 hours) improve muscle recovery.",
        "source": "General Fitness"
    },
    {
        "id": "doc4",
        "content": "For cardio endurance, try interval training like HIIT.",
        "source": "Workout Advice"
    },
    {
        "id": "doc5",
        "content": "Warm up with dynamic stretches before running to reduce injury risk.",
        "source": "Workout Advice"
    },
    {
        "id": "doc6",
        "content": "Balanced diets typically include protein, whole grains, fruits, vegetables, and healthy fats.",
        "source": "Nutrition"
    },
]
print("Created a small list of health tips.")

## 3.0. Create or Reset the Index
When creating a vector field in Azure AI Search, the **field definition** must include a `vector_search_profile` property that points to a matching profile name in your vector search settings.

We'll define a helper function to create (or reset) a vector index with an HNSW algorithm config.


In [ ]:
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    SimpleField,
    SearchableField,
    VectorSearch,
    HnswAlgorithmConfiguration,
    HnswParameters,
    VectorSearchAlgorithmKind,
    VectorSearchAlgorithmMetric,
    VectorSearchProfile,
)

def create_healthtips_index(
        endpoint: str, api_key: str, index_name: str, 
        dimension: int = 1536 # if using text-embedding-3-large
        ):
    """Create or update a search index for health tips with vector search capability."""
    
    index_client = SearchIndexClient(endpoint=endpoint, credential=AzureKeyCredential(api_key))
    
    # Try to delete existing index
    try:
        index_client.delete_index(index_name)
        print(f"Deleted existing index: {index_name}")
    except Exception:
        pass  # Index doesn't exist yet
        
    # Define vector search configuration
    vector_search = VectorSearch(
        algorithms=[
            HnswAlgorithmConfiguration(
                name="myHnsw",
                kind=VectorSearchAlgorithmKind.HNSW,
                parameters=HnswParameters(
                    m=4,
                    ef_construction=400,
                    ef_search=500,
                    metric=VectorSearchAlgorithmMetric.COSINE
                )
            )
        ],
        profiles=[
            VectorSearchProfile(
                name="myHnswProfile",
                algorithm_configuration_name="myHnsw"
            )
        ]
    )
    
    # Define fields
    fields = [
        SimpleField(name="id", type=SearchFieldDataType.String, key=True),
        SearchableField(name="content", type=SearchFieldDataType.String),
        SimpleField(name="source", type=SearchFieldDataType.String),
        SearchField(
            name="embedding", 
            type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
            vector_search_dimensions=dimension,
            vector_search_profile_name="myHnswProfile" 
        ),
    ]
    
    # Create index definition
    index_def = SearchIndex(
        name=index_name,
        fields=fields,
        vector_search=vector_search
    )
    
    # Create the index
    index_client.create_index(index_def)
    print(f"✅ Created or reset index: {index_name}")

## 3.1. Create Index & Upload Health Tips 🏋️

Now we'll put our health tips into action by:
1. **Creating a search connection** to Azure AI Search
2. **Building our index** with vector search capability
3. **Generating embeddings** for each health tip
4. **Uploading** the tips with their embeddings

This creates our knowledge base that we'll search through later. Think of it as building our 'fitness library' that our AI assistant can reference! 📚💪

In [ ]:
from azure.ai.projects.models import ConnectionType

# Create OpenAI client for embeddings and chat
client = OpenAI(
    api_key=os.getenv("AZURE_OPENAI_KEY"),
    base_url=os.getenv("AZURE_OPENAI_ENDPOINT")
)

# ── 1. Get Azure AI Search connection from Foundry project ─────────────────
search_endpoint = None
search_key = None

try:
    foundry_token = credential.get_token("https://ai.azure.com/.default").token
    mgmt_token    = credential.get_token("https://management.azure.com/.default").token
    _foundry_headers = {"Authorization": f"Bearer {foundry_token}"}
    _mgmt_headers    = {"Authorization": f"Bearer {mgmt_token}"}
    project_endpoint = os.getenv("PROJECT_ENDPOINT")

    # Get the search connection (endpoint + resource ID)
    conn_resp = requests.get(
        f"{project_endpoint}/connections/aisearch2325236b993ed"
        "?api-version=2025-05-15-preview&includeCredentials=true",
        headers=_foundry_headers, timeout=15
    )
    if conn_resp.status_code == 200:
        conn_data = conn_resp.json()
        search_endpoint = conn_data.get("target", "").rstrip("/")
        
        # The API doesn't return the key directly — get it via ARM listAdminKeys
        search_resource_id = conn_data.get("metadata", {}).get("ResourceId")
        if search_resource_id:
            keys_resp = requests.post(
                f"https://management.azure.com{search_resource_id}/listAdminKeys"
                "?api-version=2020-08-01",
                headers=_mgmt_headers, timeout=15
            )
            if keys_resp.status_code == 200:
                search_key = keys_resp.json().get("primaryKey")

except Exception as e:
    print(f"⚠️  Connection lookup error: {e}")

# Fallback to environment variables
if not (search_endpoint and search_key):
    search_endpoint = os.getenv("AZURE_SEARCH_ENDPOINT")
    search_key      = os.getenv("AZURE_SEARCH_KEY")

if not (search_endpoint and search_key):
    raise EnvironmentError(
        "Azure AI Search credentials not found. "
        "Add AZURE_SEARCH_ENDPOINT and AZURE_SEARCH_KEY to your .env file."
    )

print("✅ Got search connection")

# ── 2. Test embeddings client ──────────────────────────────────────────────
print("✅ Created embeddings client")
test_emb = client.embeddings.create(model=embedding_model, input=["test"])
embedding_dim = len(test_emb.data[0].embedding)
print(f"✅ Got embedding length: {embedding_dim}")

# ── 3. Create / reset the search index ────────────────────────────────────
create_healthtips_index(
    endpoint=search_endpoint,
    api_key=search_key,
    index_name=search_index_name,
    dimension=embedding_dim
)

# ── 4. Create search client ────────────────────────────────────────────────
search_client = SearchClient(
    endpoint=search_endpoint,
    index_name=search_index_name,
    credential=AzureKeyCredential(search_key)
)
print("✅ Created search client")

# ── 5. Embed each health tip and upload to index ───────────────────────────
search_docs = []
for doc in health_tips:
    emb_response = client.embeddings.create(
        model=embedding_model,
        input=[doc["content"]]
    )
    emb_vec = emb_response.data[0].embedding

    search_docs.append({
        "id": doc["id"],
        "content": doc["content"],
        "source": doc["source"],
        "embedding": emb_vec,
    })

result = search_client.upload_documents(documents=search_docs)
print(f"✅ Uploaded {len(search_docs)} documents to search index '{search_index_name}'")

## 4. Basic RAG Flow
### 4.1. Retrieve
When a user queries, we:
1. Embed user question.
2. Search vector index with that embedding to get top docs.

### 4.2. Generate answer
We then pass the retrieved docs to the chat model.

> In a real scenario, you'd have a more advanced approach to chunking & summarizing. We'll keep it simple.


In [ ]:
from azure.search.documents.models import VectorizedQuery

def rag_chat(query: str, top_k: int = 3) -> str:
    """RAG pipeline: embed query → Azure AI Search → generate answer"""
    # 1) Embed user query
    print(f"  Embedding query: '{query}'...")
    embedding_response = client.embeddings.create(
        model=embedding_model,
        input=[query]
    )
    user_vec = embedding_response.data[0].embedding

    # 2) Retrieve top docs from Azure AI Search
    print("  Searching Azure AI Search...")
    vector_query = VectorizedQuery(
        vector=user_vec,
        k_nearest_neighbors=top_k,
        fields="embedding"
    )
    results = search_client.search(
        search_text="",
        vector_queries=[vector_query],
        select=["content", "source"]
    )
    top_docs_content = [
        f"Source: {r['source']} => {r['content']}" for r in results
    ]

    # 3) Build grounded system prompt
    system_text = (
        "You are a health & fitness assistant.\n"
        "Answer user questions using ONLY the text from these docs.\n"
        "Docs:\n"
        + "\n".join(top_docs_content)
        + "\nIf unsure, say 'I'm not sure'.\n"
    )

    # 4) Generate answer
    print("  Generating response...")
    response = client.chat.completions.create(
        model=chat_model,
        messages=[
            {"role": "system", "content": system_text},
            {"role": "user",   "content": query}
        ]
    )
    return response.choices[0].message.content

## 5. Try a Query 🎉
Let's do a question about cardio for busy people.


In [ ]:
user_query = "What's a good short cardio routine for me if I'm busy?"
answer = rag_chat(user_query)
print("🗣️ User Query:", user_query)
print("🤖 RAG Answer:", answer)

## 6. Conclusion
We've demonstrated a **basic RAG** pipeline with:
- **Embedding** docs & storing them in **Azure AI Search**.
- **Retrieving** top docs for user question.
- **Chat** with the retrieved docs.

🔎 You can expand this by adding advanced chunking, more robust retrieval, and quality checks. Enjoy your healthy coding! 🍎


🚀 Want to optimize this further with a small language model? Check out the next notebook [4-phi-4.ipynb](4-phi-4.ipynb) to see how to use Phi-4 using the same Azure AI Foundry SDKs!